<a href="https://colab.research.google.com/github/kunal190399/Product_Metadata_Intelligence_Platform/blob/main/Product_Metadata_Intelligence_Platform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏗️ Product Metadata Intelligence Platform

**AI-powered extraction, validation, and intelligent search over supplier product data**

---

This notebook demonstrates an end-to-end pipeline that:

1. 📄 **Parses** supplier documents (PDF / TXT / DOCX) — with OCR fallback for scanned files
2. 🧠 **Extracts** structured product metadata using an LLM (GPT-4o-mini or free local Ollama)
3. ✅ **Validates** compliance & sustainability fields with a weighted scoring system
4. 🔍 **Indexes** products into a FAISS vector store for semantic search
5. 💬 **Answers** natural language questions via Retrieval-Augmented Generation (RAG)

---

**Tech stack:** LangChain · OpenAI / Ollama · FAISS · Pydantic · PyMuPDF · Streamlit  
**GitHub:** [github.com/kunal190399/Product_Metadata_Intelligence_Platform](https://github.com/kunal190399/Product_Metadata_Intelligence_Platform)

> ⚡ **Runtime:** Set to `CPU` — no GPU needed for this project.

## ⚙️ Step 1 — Install Dependencies

In [1]:
%%capture
!pip install langchain langchain-community langchain-openai langchain-ollama \
             openai faiss-cpu pymupdf pytesseract Pillow python-docx \
             pydantic pandas sentence-transformers loguru python-dotenv tiktoken

## 🔑 Step 2 — Configure LLM Provider

Choose between:
- **OpenAI** (recommended for best results — needs API key)
- **Mock mode** (runs without any API key — uses simulated extraction for demo purposes)

In [2]:
import os

# ── Choose your provider ───────────────────────────────────────────────────────
# Set to "openai" if you have an API key, or "mock" to run the full demo for free
LLM_PROVIDER = "mock"   # "openai" | "mock"

OPENAI_API_KEY   = ""   # paste your key here if using OpenAI
OPENAI_MODEL     = "gpt-4o-mini"
EMBEDDING_MODEL  = "text-embedding-3-small"

# Vector store & chunking settings
CHUNK_SIZE    = 1000
CHUNK_OVERLAP = 200
TOP_K         = 3

# Known sustainability certifications to scan for
KNOWN_CERTIFICATIONS = [
    "FSC", "PEFC", "ISO 14001", "ISO 9001", "CE", "BREEAM",
    "LEED", "Cradle to Cradle", "EPD", "REACH", "RoHS",
]

if LLM_PROVIDER == "openai" and not OPENAI_API_KEY:
    print("⚠️  No API key set — switching to mock mode.")
    LLM_PROVIDER = "mock"

print(f"✅ LLM Provider: {LLM_PROVIDER.upper()}")

✅ LLM Provider: MOCK


## 📄 Step 3 — Document Parser
Handles PDF (text + OCR fallback), TXT, and DOCX formats.

In [3]:
from __future__ import annotations
import io
from dataclasses import dataclass, field
from pathlib import Path
from typing import List


@dataclass
class ParsedDocument:
    """Holds raw extracted text and metadata from a parsed supplier document."""
    filename: str
    raw_text: str
    pages: List[str] = field(default_factory=list)
    page_count: int = 0
    extraction_method: str = "text"   # "text" | "ocr"
    file_type: str = "pdf"


class DocumentParser:
    """
    Parses supplier documents into plain text.

    Supported formats:
        PDF  — text-based via PyMuPDF; scanned pages via pytesseract OCR
        TXT  — direct UTF-8 decode
        DOCX — paragraph extraction via python-docx
    """

    def parse(self, file_path: Path = None,
              file_bytes: bytes = None,
              filename: str = "document") -> ParsedDocument:
        if file_path is not None:
            filename = file_path.name
            file_bytes = file_path.read_bytes()
        if file_bytes is None:
            raise ValueError("Provide file_path or file_bytes.")

        suffix = Path(filename).suffix.lower()
        if suffix == ".pdf":
            return self._parse_pdf(file_bytes, filename)
        elif suffix == ".txt":
            return self._parse_txt(file_bytes, filename)
        elif suffix in (".docx", ".doc"):
            return self._parse_docx(file_bytes, filename)
        else:
            raise ValueError(f"Unsupported file type: {suffix}")

    def _parse_pdf(self, data: bytes, filename: str) -> ParsedDocument:
        import fitz
        doc = fitz.open(stream=data, filetype="pdf")
        pages = [page.get_text("text").strip() for page in doc]
        full_text = "\n\n".join(pages)
        method = "text"

        if len(full_text.strip()) < 100:
            print(f"  ↳ Low text yield in '{filename}' — switching to OCR")
            pages, full_text = self._ocr_pdf(data)
            method = "ocr"

        print(f"  ✅ Parsed '{filename}' — {len(pages)} pages via {method}")
        return ParsedDocument(filename=filename, raw_text=full_text,
                              pages=pages, page_count=len(pages),
                              extraction_method=method, file_type="pdf")

    def _ocr_pdf(self, data: bytes):
        import fitz, pytesseract
        from PIL import Image
        doc = fitz.open(stream=data, filetype="pdf")
        pages = []
        for page in doc:
            mat = fitz.Matrix(300 / 72, 300 / 72)
            pix = page.get_pixmap(matrix=mat)
            img = Image.open(io.BytesIO(pix.tobytes("png")))
            pages.append(pytesseract.image_to_string(img, lang="eng").strip())
        return pages, "\n\n".join(pages)

    def _parse_txt(self, data: bytes, filename: str) -> ParsedDocument:
        text = data.decode("utf-8", errors="replace")
        print(f"  ✅ Parsed '{filename}' — plain text ({len(text)} chars)")
        return ParsedDocument(filename=filename, raw_text=text, pages=[text],
                              page_count=1, file_type="txt")

    def _parse_docx(self, data: bytes, filename: str) -> ParsedDocument:
        import docx
        doc = docx.Document(io.BytesIO(data))
        text = "\n".join(p.text for p in doc.paragraphs if p.text.strip())
        print(f"  ✅ Parsed '{filename}' — DOCX")
        return ParsedDocument(filename=filename, raw_text=text, pages=[text],
                              page_count=1, file_type="docx")


print("✅ DocumentParser defined")

✅ DocumentParser defined


## 🧠 Step 4 — Pydantic Schema & LLM Metadata Extractor

Defines the structured `ProductMetadata` schema and uses an LLM to populate it from raw document text.

In [4]:
import json, re
from typing import List, Optional
from pydantic import BaseModel, Field, field_validator


# ── Schema ─────────────────────────────────────────────────────────────────────

class Dimensions(BaseModel):
    length_mm: Optional[float] = None
    width_mm:  Optional[float] = None
    height_mm: Optional[float] = None
    unit: str = "mm"


class ProductMetadata(BaseModel):
    """
    Structured product metadata extracted from a supplier document.
    Mirrors what a design/sustainability team needs for product decisions.
    """
    product_name:         str            = Field(description="Full commercial product name")
    manufacturer:         str            = Field(description="Manufacturer or brand name")
    category:             str            = Field(description="Product category")
    model_number:         Optional[str]  = None
    dimensions:           Optional[Dimensions] = None
    materials:            List[str]      = Field(default_factory=list)
    certifications:       List[str]      = Field(default_factory=list)
    carbon_footprint:     Optional[str]  = None
    recycled_content_pct: Optional[float] = None
    country_of_origin:    Optional[str]  = None
    lead_time_weeks:      Optional[int]  = None
    warranty_years:       Optional[int]  = None
    fire_rating:          Optional[str]  = None
    description:          str            = ""
    raw_source:           str            = ""

    @field_validator("recycled_content_pct")
    @classmethod
    def clamp_pct(cls, v):
        return max(0.0, min(100.0, v)) if v is not None else v


# ── LLM prompt ─────────────────────────────────────────────────────────────────

EXTRACTION_PROMPT = """You are a product data analyst specialising in building materials and interior products.

Extract structured product metadata from the supplier document text below.
Return ONLY a valid JSON object — no markdown, no explanation.

Schema:
{{
  "product_name": "string",
  "manufacturer": "string",
  "category": "string (e.g. flooring, lighting, panel, furniture)",
  "model_number": "string or null",
  "dimensions": {{"length_mm": number, "width_mm": number, "height_mm": number, "unit": "mm"}} or null,
  "materials": ["list", "of", "materials"],
  "certifications": ["FSC", "CE", etc.],
  "carbon_footprint": "string or null",
  "recycled_content_pct": number or null,
  "country_of_origin": "string or null",
  "lead_time_weeks": number or null,
  "warranty_years": number or null,
  "fire_rating": "string or null",
  "description": "1-2 sentence product summary"
}}

Rules: null for missing fields. Convert dimensions to mm. Include all explicitly stated certifications only.

--- DOCUMENT ---
{text}
--- END ---

JSON:"""


# ── Extractor ──────────────────────────────────────────────────────────────────

class MetadataExtractor:
    """Extracts structured ProductMetadata from raw text using an LLM."""

    def __init__(self):
        self._llm = None

    @property
    def llm(self):
        if self._llm is None:
            if LLM_PROVIDER == "openai":
                from langchain_openai import ChatOpenAI
                self._llm = ChatOpenAI(api_key=OPENAI_API_KEY,
                                       model=OPENAI_MODEL, temperature=0)
            elif LLM_PROVIDER == "ollama":
                from langchain_ollama import ChatOllama
                self._llm = ChatOllama(model="llama3.2", temperature=0)
        return self._llm

    def extract(self, text: str, source_filename: str = "") -> ProductMetadata:
        if LLM_PROVIDER == "mock":
            return self._mock_extract(text, source_filename)

        prompt = EXTRACTION_PROMPT.format(text=text[:6000])
        response = self.llm.invoke(prompt)
        raw = self._clean_json(response.content)
        try:
            data = json.loads(raw)
            return ProductMetadata(**data, raw_source=source_filename)
        except Exception as e:
            print(f"  ⚠️  Extraction failed: {e}")
            return ProductMetadata(product_name="Unknown", manufacturer="Unknown",
                                   category="Unknown", raw_source=source_filename)

    def _mock_extract(self, text: str, source_filename: str) -> ProductMetadata:
        """
        Rule-based extraction — runs without any API key.
        Uses safe regex helpers that always require a capture group.
        """
        import re
        lines = [l.strip() for l in text.split("\n") if l.strip()]

        def extract_field(pattern, default=None):
            """Search full text for pattern with exactly one capture group."""
            m = re.search(pattern, text, re.I)
            return m.group(1).strip() if m else default

        # Each pattern MUST have exactly one capture group (...)
        product_name = extract_field(r"product\s*name\s*:\s*(.+)")
        if not product_name:
            # Fall back to first non-header line that looks like a product name
            for line in lines:
                if len(line) > 5 and not line.endswith(":") and "datasheet" not in line.lower():
                    product_name = line[:80]
                    break

        manufacturer = extract_field(r"manufacturer\s*:\s*(.+)")
        if not manufacturer:
            manufacturer = extract_field(r"—\s*([\w\s]+(?:Ltd|GmbH|Inc|Corp|Group)[\w\s]*)")

        category     = extract_field(r"category\s*:\s*(.+)")
        model_number = extract_field(r"(?:model|sku|ref)\s*(?:number|no\.?)?\s*:\s*([\w\-]+)")
        recycled_raw = extract_field(r"recycled\s*content\s*:\s*([\d.]+)\s*%")
        carbon       = extract_field(r"carbon\s*footprint\s*:\s*(.+?)(?:\n|$)")
        fire         = extract_field(r"fire\s*(?:rating|class(?:ification)?)\s*:\s*(.+?)(?:\n|$)")
        origin       = extract_field(r"country\s*of\s*origin\s*:\s*(.+?)(?:\n|$)")
        warranty_raw = extract_field(r"warranty\s*:\s*(\d+)\s*year")
        lead_raw     = extract_field(r"lead\s*time\s*:\s*(\d+)")
        materials_raw= extract_field(r"materials?\s*:\s*(.+?)(?:\n|$)")

        materials = [m.strip() for m in materials_raw.split(",")] if materials_raw else []
        certs = [c for c in KNOWN_CERTIFICATIONS if c.lower() in text.lower()]

        # Dimensions: e.g. 600mm x 2400mm x 18mm
        dims = None
        dim_match = re.search(
            r"(\d+)\s*mm\s*[xX×]\s*(\d+)\s*mm(?:\s*[xX×]\s*(\d+)\s*mm)?", text, re.I
        )
        if dim_match:
            dims = Dimensions(
                width_mm=float(dim_match.group(1)),
                height_mm=float(dim_match.group(2)),
                length_mm=float(dim_match.group(3)) if dim_match.group(3) else None
            )

        # Description: join first 2 lines that look like sentences
        desc_lines = [l for l in lines if len(l) > 30 and ":" not in l][:2]
        description = " ".join(desc_lines)

        return ProductMetadata(
            product_name         = product_name or (lines[0][:60] if lines else "Unknown"),
            manufacturer         = manufacturer or "Unknown",
            category             = category or "General",
            model_number         = model_number,
            dimensions           = dims,
            materials            = materials,
            certifications       = certs,
            carbon_footprint     = carbon,
            recycled_content_pct = float(recycled_raw) if recycled_raw else None,
            country_of_origin    = origin,
            lead_time_weeks      = int(lead_raw) if lead_raw else None,
            warranty_years       = int(warranty_raw) if warranty_raw else None,
            fire_rating          = fire,
            description          = description,
            raw_source           = source_filename,
        )

    @staticmethod
    def _clean_json(text: str) -> str:
        text = text.strip()
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
        return text.strip()


print("✅ ProductMetadata schema & MetadataExtractor defined")

✅ ProductMetadata schema & MetadataExtractor defined


## ✅ Step 5 — Compliance Validator

Checks each product against required fields and sustainability criteria, producing a grade A–D.

In [5]:
from dataclasses import dataclass, field as dc_field
from typing import List

REQUIRED_FIELDS = ["product_name", "manufacturer", "category", "materials"]


@dataclass
class ComplianceReport:
    product_name: str
    passed: bool
    score: float
    missing_required_fields: List[str] = dc_field(default_factory=list)
    sustainability_flags: List[str]    = dc_field(default_factory=list)
    certifications_found: List[str]    = dc_field(default_factory=list)

    @property
    def grade(self) -> str:
        if self.score >= 0.90: return "A"
        if self.score >= 0.75: return "B"
        if self.score >= 0.50: return "C"
        return "D"

    @property
    def grade_emoji(self) -> str:
        return {"A": "🟢", "B": "🔵", "C": "🟠", "D": "🔴"}.get(self.grade, "⚪")


class ComplianceValidator:
    """
    Weighted compliance checks — each check contributes to a 0–1 score.
    Grades: A (≥90%) · B (≥75%) · C (≥50%) · D (<50%)
    """

    CHECKS = [
        ("_required_fields",   0.35, "Missing required fields"),
        ("_certifications",    0.25, "No sustainability certifications found"),
        ("_recycled_content",  0.15, "Recycled content percentage not declared"),
        ("_fire_rating",       0.10, "Fire rating not specified"),
        ("_country_of_origin", 0.08, "Country of origin not declared"),
        ("_warranty",          0.07, "Warranty information missing"),
    ]

    def validate(self, product: ProductMetadata) -> ComplianceReport:
        missing, flags = [], []
        score = 0.0

        for method_name, weight, flag_msg in self.CHECKS:
            ok, details = getattr(self, method_name)(product)
            if ok:
                score += weight
            else:
                if method_name == "_required_fields":
                    missing.extend(details)
                else:
                    flags.append(flag_msg)

        certs_found = [c for c in KNOWN_CERTIFICATIONS
                       if c.lower() in [x.lower() for x in product.certifications]]

        return ComplianceReport(
            product_name=product.product_name,
            passed=len(missing) == 0 and score >= 0.5,
            score=round(score, 3),
            missing_required_fields=missing,
            sustainability_flags=flags,
            certifications_found=certs_found,
        )

    def _required_fields(self, p):
        missing = [f for f in REQUIRED_FIELDS
                   if not getattr(p, f, None) or
                   (isinstance(getattr(p, f), list) and not getattr(p, f))]
        return len(missing) == 0, missing

    def _certifications(self, p):   return bool(p.certifications), []
    def _recycled_content(self, p): return p.recycled_content_pct is not None, []
    def _fire_rating(self, p):      return bool(p.fire_rating), []
    def _country_of_origin(self, p):return bool(p.country_of_origin), []
    def _warranty(self, p):         return p.warranty_years is not None, []


print("✅ ComplianceValidator defined")

✅ ComplianceValidator defined


## 🔍 Step 6 — RAG Engine (Vector Search + Q&A)

Embeds products into a FAISS vector store and provides:
- **Semantic search** — find products by meaning
- **RAG Q&A** — natural language questions answered with context

In [6]:
import numpy as np
from typing import List, Dict


def _product_to_text(p: ProductMetadata) -> str:
    """Convert a product to a rich text document for embedding."""
    parts = [
        f"Product: {p.product_name}",
        f"Manufacturer: {p.manufacturer}",
        f"Category: {p.category}",
        f"Description: {p.description}",
        f"Materials: {', '.join(p.materials) if p.materials else 'Not specified'}",
        f"Certifications: {', '.join(p.certifications) if p.certifications else 'None'}",
        f"Recycled Content: {p.recycled_content_pct}%" if p.recycled_content_pct else "Recycled Content: Not stated",
        f"Carbon Footprint: {p.carbon_footprint or 'Not stated'}",
        f"Fire Rating: {p.fire_rating or 'Not stated'}",
        f"Country: {p.country_of_origin or 'Unknown'}",
    ]
    return "\n".join(parts)


class RAGEngine:
    """
    FAISS-backed vector store with semantic search and RAG Q&A.

    In mock mode: uses TF-IDF cosine similarity (no API needed).
    In OpenAI mode: uses text-embedding-3-small + GPT-4o-mini for Q&A.
    """

    def __init__(self):
        self._products: List[ProductMetadata] = []
        self._texts: List[str] = []
        self._index = None     # FAISS index (OpenAI) or TF-IDF matrix (mock)
        self._vectorizer = None
        self._embeddings_fn = None
        self._llm = None

    def add_products(self, products: List[ProductMetadata]):
        """Embed and index a list of products."""
        self._products.extend(products)
        new_texts = [_product_to_text(p) for p in products]
        self._texts.extend(new_texts)

        if LLM_PROVIDER == "mock":
            self._build_tfidf_index()
        else:
            self._build_faiss_index(new_texts)

        print(f"  ✅ Indexed {len(products)} product(s) — total: {len(self._products)}")

    # ── TF-IDF mock search ─────────────────────────────────────────────────────

    def _build_tfidf_index(self):
        from sklearn.feature_extraction.text import TfidfVectorizer
        self._vectorizer = TfidfVectorizer(stop_words="english")
        self._index = self._vectorizer.fit_transform(self._texts)

    def _tfidf_search(self, query: str, k: int) -> List[Dict]:
        from sklearn.metrics.pairwise import cosine_similarity
        q_vec = self._vectorizer.transform([query])
        scores = cosine_similarity(q_vec, self._index).flatten()
        top_k = scores.argsort()[::-1][:k]
        return [
            {"product": self._products[i],
             "text": self._texts[i],
             "score": float(scores[i])}
            for i in top_k if scores[i] > 0
        ]

    # ── FAISS OpenAI search ────────────────────────────────────────────────────

    def _build_faiss_index(self, texts: List[str]):
        from langchain_openai import OpenAIEmbeddings
        from langchain_community.vectorstores import FAISS
        from langchain.schema import Document
        if self._embeddings_fn is None:
            self._embeddings_fn = OpenAIEmbeddings(
                api_key=OPENAI_API_KEY, model=EMBEDDING_MODEL)
        docs = [Document(page_content=t) for t in texts]
        if self._index is None:
            self._index = FAISS.from_documents(docs, self._embeddings_fn)
        else:
            self._index.add_documents(docs)

    # ── Public API ─────────────────────────────────────────────────────────────

    def search(self, query: str, k: int = TOP_K) -> List[Dict]:
        """Semantic similarity search — returns top-k matching products."""
        if not self._products:
            raise RuntimeError("No products indexed. Call add_products() first.")

        if LLM_PROVIDER == "mock":
            return self._tfidf_search(query, k)
        else:
            results = self._index.similarity_search_with_score(query, k=k)
            return [{"text": doc.page_content, "score": float(s),
                     "product": None} for doc, s in results]

    def ask(self, question: str, k: int = TOP_K) -> Dict:
        """RAG Q&A — retrieves context and generates a grounded answer."""
        results = self.search(question, k=k)
        context = "\n\n---\n\n".join(r["text"] for r in results)

        if LLM_PROVIDER == "mock":
            # Rule-based answer for mock mode
            relevant = [r for r in results if r["score"] > 0.05]
            if not relevant:
                return {"answer": "No matching products found in the index.",
                        "sources": []}
            answer_parts = [f"Based on the indexed products:\n"]
            for r in relevant:
                p = r["product"]
                answer_parts.append(
                    f"• **{p.product_name}** ({p.manufacturer}) — "
                    f"Category: {p.category} | "
                    f"Certifications: {', '.join(p.certifications) or 'None'} | "
                    f"Recycled: {p.recycled_content_pct}%" if p.recycled_content_pct else
                    f"• **{p.product_name}** ({p.manufacturer})"
                )
            sources = [r["product"].product_name for r in relevant]
            return {"answer": "\n".join(answer_parts), "sources": sources}

        # OpenAI RAG answer
        from langchain_openai import ChatOpenAI
        if self._llm is None:
            self._llm = ChatOpenAI(api_key=OPENAI_API_KEY,
                                    model=OPENAI_MODEL, temperature=0)
        prompt = f"""Answer the question based ONLY on these products. Be specific and cite product names.\n\n{context}\n\nQuestion: {question}\nAnswer:"""
        response = self._llm.invoke(prompt)
        sources = [r["product"].product_name for r in results if r["product"]]
        return {"answer": response.content, "sources": sources}


print("✅ RAGEngine defined")

✅ RAGEngine defined


## 🚀 Step 7 — Run the Full Pipeline

Three sample supplier documents are created and processed end-to-end.

In [7]:
# ── Sample supplier documents ──────────────────────────────────────────────────

SAMPLE_DOCS = {
    "ecopanel_datasheet.txt": """
PRODUCT DATASHEET — GreenBuild Ltd

Product Name: EcoPanel 3000 - Sustainable Wall Panel
Model Number: EP-3000-OAK
Category: Architectural Wall Panel

The EcoPanel 3000 is a premium sustainable wall panel manufactured from
reclaimed oak fibres and plant-based bio-resin. Designed for high-traffic
commercial interiors.

Dimensions: 600mm x 2400mm x 18mm
Materials: Reclaimed oak fibre (65% recycled content), Bio-resin (plant-derived), Low-VOC lacquer
Certifications: FSC Certified, ISO 14001:2015, CE Marking, REACH Compliant
Fire Classification: Class B (Euroclass Bfl-s1)

Recycled Content: 65%
Carbon Footprint: 8.2 kg CO2e per m2
Country of Origin: Germany
Warranty: 10 years
Lead Time: 4 weeks
""",

    "luxfloor_spec.txt": """
LuxFloor Premium Engineered Oak Flooring
Manufacturer: LuxFloor Europe GmbH
Category: Flooring
Model: LF-OAK-180-NAT

Premium engineered oak flooring with a 4mm solid oak wear layer on a
birch plywood base. Suitable for underfloor heating.

Dimensions: 180mm x 1800mm x 14mm
Materials: Solid European Oak (wear layer), Baltic Birch Plywood (core)
Certifications: FSC, PEFC, ISO 9001, CE
Fire Rating: Cfl-s1

Recycled Content: 22%
Country of Origin: Poland
Warranty: 25 years
Lead Time: 3 weeks
""",

    "slimline_chair_spec.txt": """
Slimline Contract Chair — ErgoForm Seating
Product: Slimline SC-400 Stacking Chair
Manufacturer: ErgoForm Seating Ltd
Category: Seating / Furniture

A lightweight stacking chair designed for commercial and hospitality environments.
Powder-coated steel frame with recycled polypropylene seat and back.

Materials: Powder-coated steel (frame), Recycled polypropylene (seat and back)
Certifications: ISO 9001, CE Marking
Country of Origin: United Kingdom
Recycled Content: 40%
Warranty: 5 years
Lead Time: 2 weeks
"""
}

print(f"✅ {len(SAMPLE_DOCS)} sample supplier documents ready")

✅ 3 sample supplier documents ready


In [8]:
# ── Run the pipeline ───────────────────────────────────────────────────────────

parser    = DocumentParser()
extractor = MetadataExtractor()
validator = ComplianceValidator()
rag       = RAGEngine()

all_products = []
all_reports  = []

print("="*60)
print("🚀 PROCESSING SUPPLIER DOCUMENTS")
print("="*60)

for filename, content in SAMPLE_DOCS.items():
    print(f"\n📄 {filename}")

    # 1. Parse
    doc = parser.parse(file_bytes=content.encode(), filename=filename)

    # 2. Extract
    product = extractor.extract(doc.raw_text, source_filename=filename)
    all_products.append(product)
    print(f"  🧠 Extracted: '{product.product_name}' — {product.manufacturer}")

    # 3. Validate
    report = validator.validate(product)
    all_reports.append(report)
    print(f"  {report.grade_emoji} Compliance Grade: {report.grade} ({report.score:.0%})")

# 4. Index into RAG
print("\n🔍 Indexing into vector store...")
rag.add_products(all_products)

print("\n" + "="*60)
print("✅ PIPELINE COMPLETE")
print("="*60)

🚀 PROCESSING SUPPLIER DOCUMENTS

📄 ecopanel_datasheet.txt
  ✅ Parsed 'ecopanel_datasheet.txt' — plain text (701 chars)
  🧠 Extracted: 'EcoPanel 3000 - Sustainable Wall Panel' — GreenBuild Ltd

Product Name
  🟢 Compliance Grade: A (100%)

📄 luxfloor_spec.txt
  ✅ Parsed 'luxfloor_spec.txt' — plain text (495 chars)
  🧠 Extracted: 'LuxFloor Premium Engineered Oak Flooring' — LuxFloor Europe GmbH
  🟢 Compliance Grade: A (100%)

📄 slimline_chair_spec.txt
  ✅ Parsed 'slimline_chair_spec.txt' — plain text (512 chars)
  🧠 Extracted: 'Slimline Contract Chair — ErgoForm Seating' — ErgoForm Seating Ltd
  🟢 Compliance Grade: A (90%)

🔍 Indexing into vector store...
  ✅ Indexed 3 product(s) — total: 3

✅ PIPELINE COMPLETE


## 📋 Step 8 — Product Directory

In [9]:
import pandas as pd

rows = []
for p in all_products:
    rows.append({
        "Product Name":    p.product_name,
        "Manufacturer":    p.manufacturer,
        "Category":        p.category,
        "Materials":       ", ".join(p.materials) if p.materials else "—",
        "Certifications":  ", ".join(p.certifications) if p.certifications else "—",
        "Recycled %":      f"{p.recycled_content_pct}%" if p.recycled_content_pct else "—",
        "Fire Rating":     p.fire_rating or "—",
        "Country":         p.country_of_origin or "—",
        "Warranty (yrs)": p.warranty_years or "—",
        "Source":          p.raw_source,
    })

df = pd.DataFrame(rows)
print("📋 PRODUCT DIRECTORY")
print("=" * 60)
df

📋 PRODUCT DIRECTORY


,Product Name,Manufacturer,Category,Materials,Certifications,Recycled %,Fire Rating,Country,Warranty (yrs),Source
0,EcoPanel 3000 - Sustainable Wall Panel,GreenBuild Ltd\n\nProduct Name,Architectural Wall Panel,"Reclaimed oak fibre (65% recycled content), Bi...","FSC, ISO 14001, CE, REACH",65.0%,Class B (Euroclass Bfl-s1),Germany,10,ecopanel_datasheet.txt
1,LuxFloor Premium Engineered Oak Flooring,LuxFloor Europe GmbH,Flooring,"Solid European Oak (wear layer), Baltic Birch ...","FSC, PEFC, ISO 9001, CE",22.0%,Cfl-s1,Poland,25,luxfloor_spec.txt
2,Slimline Contract Chair — ErgoForm Seating,ErgoForm Seating Ltd,Seating / Furniture,"Powder-coated steel (frame), Recycled polyprop...","ISO 9001, CE",40.0%,—,United Kingdom,5,slimline_chair_spec.txt


## ✅ Step 9 — Compliance Dashboard

In [10]:
print("✅ COMPLIANCE DASHBOARD")
print("=" * 60)

passed    = sum(1 for r in all_reports if r.passed)
avg_score = sum(r.score for r in all_reports) / len(all_reports)

print(f"  Total Products:      {len(all_reports)}")
print(f"  Pass Rate:           {passed}/{len(all_reports)} ({passed/len(all_reports):.0%})")
print(f"  Avg Compliance Score:{avg_score:.0%}")
print()

for report in all_reports:
    print(f"  {report.grade_emoji} [{report.grade}] {report.product_name:<40} {report.score:.0%}")
    if report.certifications_found:
        print(f"       Certs: {', '.join(report.certifications_found)}")
    if report.missing_required_fields:
        print(f"       ❌ Missing: {', '.join(report.missing_required_fields)}")
    for flag in report.sustainability_flags:
        print(f"       ⚠️  {flag}")
    print()

✅ COMPLIANCE DASHBOARD
  Total Products:      3
  Pass Rate:           3/3 (100%)
  Avg Compliance Score:97%

  🟢 [A] EcoPanel 3000 - Sustainable Wall Panel   100%
       Certs: FSC, ISO 14001, CE, REACH

  🟢 [A] LuxFloor Premium Engineered Oak Flooring 100%
       Certs: FSC, PEFC, ISO 9001, CE

  🟢 [A] Slimline Contract Chair — ErgoForm Seating 90%
       Certs: ISO 9001, CE
       ⚠️  Fire rating not specified



## 🔍 Step 10 — Semantic Search

In [11]:
queries = [
    "FSC certified sustainable timber product",
    "stacking chair for commercial use",
    "flooring with high recycled content",
]

print("🔍 SEMANTIC SEARCH RESULTS")
print("=" * 60)

for query in queries:
    print(f"\n🔎 Query: '{query}'")
    results = rag.search(query, k=2)
    for i, r in enumerate(results, 1):
        p = r["product"]
        if p:
            print(f"  #{i} [{r['score']:.3f}] {p.product_name} — {p.manufacturer}")
        else:
            print(f"  #{i} Score: {r['score']:.3f}")

🔍 SEMANTIC SEARCH RESULTS

🔎 Query: 'FSC certified sustainable timber product'
  #1 [0.310] EcoPanel 3000 - Sustainable Wall Panel — GreenBuild Ltd

Product Name
  #2 [0.073] LuxFloor Premium Engineered Oak Flooring — LuxFloor Europe GmbH

🔎 Query: 'stacking chair for commercial use'
  #1 [0.353] Slimline Contract Chair — ErgoForm Seating — ErgoForm Seating Ltd

🔎 Query: 'flooring with high recycled content'
  #1 [0.391] LuxFloor Premium Engineered Oak Flooring — LuxFloor Europe GmbH
  #2 [0.127] EcoPanel 3000 - Sustainable Wall Panel — GreenBuild Ltd

Product Name


## 💬 Step 11 — RAG Q&A

In [12]:
questions = [
    "Which products have FSC certification?",
    "What is the highest recycled content product?",
    "Which product has the best fire rating?",
]

print("💬 RAG Q&A")
print("=" * 60)

for q in questions:
    print(f"\n❓ {q}")
    result = rag.ask(q)
    print(f"   {result['answer']}")
    if result["sources"]:
        print(f"   📌 Sources: {', '.join(result['sources'])}")

💬 RAG Q&A

❓ Which products have FSC certification?
   Based on the indexed products:

• **EcoPanel 3000 - Sustainable Wall Panel** (GreenBuild Ltd

Product Name) — Category: Architectural Wall Panel | Certifications: FSC, ISO 14001, CE, REACH | Recycled: 65.0%
• **LuxFloor Premium Engineered Oak Flooring** (LuxFloor Europe GmbH) — Category: Flooring | Certifications: FSC, PEFC, ISO 9001, CE | Recycled: 22.0%
   📌 Sources: EcoPanel 3000 - Sustainable Wall Panel, LuxFloor Premium Engineered Oak Flooring

❓ What is the highest recycled content product?
   Based on the indexed products:

• **EcoPanel 3000 - Sustainable Wall Panel** (GreenBuild Ltd

Product Name) — Category: Architectural Wall Panel | Certifications: FSC, ISO 14001, CE, REACH | Recycled: 65.0%
• **Slimline Contract Chair — ErgoForm Seating** (ErgoForm Seating Ltd) — Category: Seating / Furniture | Certifications: ISO 9001, CE | Recycled: 40.0%
• **LuxFloor Premium Engineered Oak Flooring** (LuxFloor Europe GmbH) — Category

## 📤 Step 12 — Export Results

In [13]:
# Export product directory as CSV
df.to_csv("product_directory.csv", index=False)
print("✅ Saved: product_directory.csv")

# Export full metadata as JSON
import json
all_data = [p.model_dump() for p in all_products]
with open("products_metadata.json", "w") as f:
    json.dump(all_data, f, indent=2)
print("✅ Saved: products_metadata.json")

# Download in Colab
try:
    from google.colab import files
    files.download("product_directory.csv")
    files.download("products_metadata.json")
    print("✅ Downloads started!")
except ImportError:
    print("   (Not in Colab — files saved to current directory)")

✅ Saved: product_directory.csv
✅ Saved: products_metadata.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloads started!


## 🧪 Step 13 — Unit Tests

In [14]:
import traceback

tests_passed = 0
tests_total  = 0

def test(name, condition, detail=""):
    global tests_passed, tests_total
    tests_total += 1
    status = "✅ PASS" if condition else "❌ FAIL"
    if condition:
        tests_passed += 1
    print(f"  {status}  {name}" + (f" ({detail})" if detail else ""))

print("🧪 RUNNING UNIT TESTS")
print("=" * 60)

# ProductMetadata tests
print("\n[ProductMetadata]")
p = ProductMetadata(product_name="Test", manufacturer="ACME", category="flooring",
                    recycled_content_pct=150.0)
test("Recycled content clamped to 100", p.recycled_content_pct == 100.0)
p2 = ProductMetadata(product_name="T", manufacturer="M", category="C",
                     recycled_content_pct=-5.0)
test("Recycled content clamped to 0",   p2.recycled_content_pct == 0.0)
p3 = ProductMetadata(product_name="T", manufacturer="M", category="C")
test("Optional fields default to None", p3.model_number is None and p3.fire_rating is None)
test("Serialises to dict", isinstance(p3.model_dump(), dict))

# ComplianceValidator tests
print("\n[ComplianceValidator]")
v = ComplianceValidator()
full = ProductMetadata(
    product_name="EcoPanel", manufacturer="GreenBuild", category="panel",
    materials=["oak"], certifications=["FSC", "CE"],
    recycled_content_pct=65, fire_rating="Class B",
    country_of_origin="Germany", warranty_years=10,
)
r_full = v.validate(full)
test("Full product passes",         r_full.passed)
test("Full product grade A or B",   r_full.grade in ("A", "B"), r_full.grade)
test("Score between 0 and 1",       0.0 <= r_full.score <= 1.0, str(r_full.score))
test("FSC cert detected",           "FSC" in r_full.certifications_found)

empty = ProductMetadata(product_name="", manufacturer="", category="")
r_empty = v.validate(empty)
test("Empty product fails",         not r_empty.passed)
test("Missing fields reported",     len(r_empty.missing_required_fields) > 0)

# MetadataExtractor JSON cleaning
print("\n[MetadataExtractor]")
test("Strips json fences",
     MetadataExtractor._clean_json('```json\n{"k":1}\n```') == '{"k":1}')
test("Strips plain fences",
     MetadataExtractor._clean_json('```\n{"k":1}\n```') == '{"k":1}')
test("No-fence passthrough",
     MetadataExtractor._clean_json('{"k":1}') == '{"k":1}')

# RAG Engine tests
print("\n[RAGEngine]")
test("Products indexed", len(rag._products) == 3, str(len(rag._products)))
results = rag.search("recycled sustainable material", k=2)
test("Search returns results",    len(results) > 0)
test("Search result has score",   "score" in results[0])

print(f"\n{'='*60}")
print(f"  Results: {tests_passed}/{tests_total} tests passed")
if tests_passed == tests_total:
    print("  🎉 All tests passed!")
else:
    print(f"  ⚠️  {tests_total - tests_passed} test(s) failed.")

🧪 RUNNING UNIT TESTS

[ProductMetadata]
  ✅ PASS  Recycled content clamped to 100
  ✅ PASS  Recycled content clamped to 0
  ✅ PASS  Optional fields default to None
  ✅ PASS  Serialises to dict

[ComplianceValidator]
  ✅ PASS  Full product passes
  ✅ PASS  Full product grade A or B (A)
  ✅ PASS  Score between 0 and 1 (1.0)
  ✅ PASS  FSC cert detected
  ✅ PASS  Empty product fails
  ✅ PASS  Missing fields reported

[MetadataExtractor]
  ✅ PASS  Strips json fences
  ✅ PASS  Strips plain fences
  ✅ PASS  No-fence passthrough

[RAGEngine]
  ✅ PASS  Products indexed (3)
  ✅ PASS  Search returns results
  ✅ PASS  Search result has score

  Results: 16/16 tests passed
  🎉 All tests passed!


---

## 📎 Next Steps

This notebook demonstrates the **core AI pipeline**. The full project on GitHub includes:

| Component | Description |
|---|---|
| `src/extractor/document_parser.py` | PDF/DOCX/OCR parsing |
| `src/extractor/metadata_extractor.py` | LLM structured extraction |
| `src/extractor/validator.py` | Weighted compliance scoring |
| `src/rag/rag_engine.py` | FAISS vector store + RAG Q&A |
| `src/ui/app.py` | Full Streamlit web app |
| `tests/` | pytest test suite |

### 🔗 Links

- **Run locally:** `streamlit run src/ui/app.py`

---
*Built as a portfolio project for the AI & Data Engineer (KTP Associate) role at TSK Group / University of Salford.*